# Damage preview & gallery

What each disaster type does to a building, and to the ground around it.

Every cell here drives [`tools/damage_gallery.py`](../tools/damage_gallery.py),
which builds a one-building scene through the **real** pipeline —
`disaster_stage.apply_to_buildings` then `mesh_damage.apply_to_stage`, off the
compiled config `compile_disaster.py` emits for that type — and then renders it
with headless Cycles. So what you see is evidence about the generator, not about
this notebook.

### Why this notebook shells out instead of importing

The build half needs `pxr` and the render half needs `bpy`, and those cannot
share an interpreter: `bpy` ships only for CPython 3.13 and statically links its
own USD (the note at the top of `disaster/mesh_damage.py` has the details). The
tool already spans that split — host `python3` for the USD, `uv run --script` for
the render — so driving it as a subprocess means this notebook runs on **any**
kernel with Pillow. The `AirStack (uv)` kernel is fine.

### What can be previewed

Only assets that are **on disk**. Nucleus (`omniverse://`) resolves in Isaac Sim
and nowhere else, so the `urban` set's buildings cannot be rendered here. The
`suburban` set can: its houses and debris are all in the `objaverse://` cache.

In [ ]:
import json, subprocess, sys
from pathlib import Path
from IPython.display import Image, display

SCENE_GEN = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TOOL = SCENE_GEN / 'tools' / 'damage_gallery.py'
OUT = SCENE_GEN / 'galleries'


def gallery(name, *args, show=True, width=1500):
    """Run the tool into galleries/<name>/ and display the sheet."""
    out = OUT / name
    cmd = ['python3', str(TOOL), '--out', str(out), *map(str, args)]
    r = subprocess.run(cmd, cwd=SCENE_GEN, capture_output=True, text=True)
    # The generator narrates heavily; keep the lines that say what was built.
    for line in r.stdout.splitlines():
        if not line.startswith('[scene_gen]'):
            print(line)
    if r.returncode:
        print(r.stderr[-3000:], file=sys.stderr)
        raise RuntimeError(f'exit {r.returncode}')
    sheet = out / 'sheet.png'
    if show and sheet.exists():
        display(Image(filename=str(sheet), width=width))
    return out


def stats(out):
    """The per-cell numbers behind a sheet, as a table."""
    man = json.loads((out / 'manifest.json').read_text())
    cols = man['columns']
    print(f"{'building':26s}" + ''.join(f'{c:>16s}' for c in cols))
    for row in man['rows']:
        line = f"{row['name'][:25]:26s}"
        for c in cols:
            s = row['stats'].get(c, {})
            line += (f"{s.get('fragments', 0):>7d}f"
                     f"{s.get('debris', 0) + s.get('debris_piles', 0):>7d}d")
        print(line)


print(TOOL.exists(), OUT)

## 1. What is available

`--list` reports which of an asset set's buildings resolve locally, and says so
when they do not rather than quietly producing a short gallery.

In [ ]:
print(subprocess.run(['python3', str(TOOL), '--list'],
                    cwd=SCENE_GEN, capture_output=True, text=True).stdout)

## 2. One building, one disaster

The fast loop while tuning a profile. `--one N` picks a building by its index
in the list above; `--disaster` restricts the columns (pristine is always kept,
because a before/after with no before is not a comparison).

Renders are a second each at this resolution, so iterate here and save the full
sheet for when a profile is worth looking at across the library.

In [ ]:
out = gallery('one', '--one', 0, '--disaster', 'tornado',
              '--severity', 0.8, '--res', 640, '--samples', 48)

## 3. The gallery

Rows are buildings, columns are disaster types, pristine on the far left.

The camera is computed **once per row, from the pristine cell**, and reused
across it. Framing each cell on its own bounds instead makes the sheet
misleading: debris widens the scene, the camera pulls back, and a *more*
damaged building renders *smaller* — the eye reads that as scale, not damage.

What to look for, per column:

| column | should read as |
|---|---|
| `earthquake` | failed in place — a storey crushed, the envelope opened, rubble at the facades |
| `tornado` | the top torn off and thrown downwind, walls standing below |
| `hurricane` | the same but shallower — roof gone, structure intact |
| `fire` | a gutted shell — roof consumed and dropped straight in, walls standing, everything charred |
| `explosion` | one breached corner, pieces thrown outward, scorched |
| `flood` | still standing. Scour at the waterline and deposited debris, nothing structural |

In [ ]:
out = gallery('damage', '--rows', 5, '--severity', 0.8,
              '--res', 520, '--samples', 48)
stats(out)

## 4. Does severity mean anything?

The load-bearing question for this whole generator: a severity sweep has to give
the *same building* at *different damage levels*, or comparing a search algorithm
across severities compares nothing.

Columns become severities of one disaster. Read left to right — the damage and
the debris should both grow, monotonically and visibly.

> This sweep is what caught two real defects. `_mesh_damage` used to carry the
> spatial damage field alone, whose core reads 1.0 at every severity, so every
> building was wrecked at full strength and only their *number* changed — the
> sweep came out as five identical ruins. And every fragment used to be handed to
> the PhysX settle pass, so gravity levelled the building whatever the severity
> was. Both are fixed; this cell is how you would notice them coming back.

In [ ]:
out = gallery('sweep_earthquake', '--sweep', 'earthquake',
              '--severities', '0.2,0.4,0.6,0.8,1.0', '--rows', 3,
              '--res', 460, '--samples', 40)
stats(out)

In [ ]:
out = gallery('sweep_hurricane', '--sweep', 'hurricane',
              '--severities', '0.2,0.5,0.8,1.0', '--rows', 3,
              '--res', 460, '--samples', 40)

## 5. Limits worth knowing

- **Settling is approximated.** `scene_prep.settle_rigid_props` drops every loose
  piece under PhysX and freezes it where it lands; that pass plays the Isaac Sim
  timeline and cannot run here. The tool rolls settling props flat and drops each
  loose piece straight down instead, which puts debris on the ground but does not
  pile it. `--no-settle` shows the authored poses — planks on end, fragments
  hanging where they were cut — which is what the sim receives before it settles.
- **Only the *loose* fragments fall.** Pieces still standing on their footings are
  left where they were cut, and that split is what makes intensity mean anything
  (`mesh_damage.fracture_to_stage`).
- **The fate is forced.** A real scene rolls damaged/destroyed/untouched per
  building against the field; a gallery pins the field to 1.0 and the fate to
  `--fate`, or a third of the sheet would come out pristine. `--fate destroyed`
  previews the ruin-asset swap instead of the mesh-damage path.
- **The takeoff-pad exclusion is cleared.** `default.yaml` keeps a 10 m clutter
  keep-out at the origin, which is exactly where a gallery stands its building —
  left in, it silently ate almost every piece of debris.
- **Charring is re-applied at render time.** `scorch` authors soot where USD
  composition consults it — `inputs:scale` on the albedo texture, because the
  shader's `diffuseColor` is *connected* and a value there is ignored. Hydra
  and Isaac honour that; Blender's USD importer drops it. So the factor rides
  through the manifest and the renderer splices the multiply back in, or the
  fire column would show a fire that chars nothing.
- **Nucleus assets cannot be previewed**, so the `urban` set's towers and
  brownstones are not in reach of this notebook. They are in reach of
  `reload_scene.py` against a live Kit instance.

Sheets and tiles land under `scene_gen/galleries/` and are gitignored; the USDs
beside them can be opened directly in Isaac Sim or `render_usd.py`.